# Paper 1 — R1 revision experiments (Colab runner)

Runs the four reviewer-response experiments (E-A model-style confound, E-B edge ablation, E-C sequence baseline, E-D AUPRC) on GPU, reusing the submitted pipeline (`scripts/graph_supervised.py`).

**Results are written directly into `experiments_R1/results/`**: four JSON files plus a consolidated `R1_RESULTS.md`.

Set the runtime to GPU first: **Runtime -> Change runtime type -> A100 (or T4)**. Then run the cells top to bottom (~10-20 min).

In [ ]:
# Cell 1 - mount Drive, cd to project, install deps
import os
from google.colab import drive; drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/mcp-agent-attack-detection'
assert os.path.isdir(ROOT), f'project not found at {ROOT} - fix ROOT to match your Drive'
os.chdir(ROOT)
import torch; print('torch', torch.__version__, '| cuda', torch.version.cuda)
# torch_geometric 2.x needs no compiled companions for GAT/GCN/SAGE; this install
# should NOT change the preinstalled torch. If the torch version printed below
# differs from above, do Runtime -> Restart session, then re-run from Cell 1.
!pip -q install torch_geometric sentence-transformers xgboost
import torch; print('torch after install:', torch.__version__, '| cuda', torch.version.cuda)

In [ ]:
# Cell 2 - run all four experiments; write result JSONs into the revision folder
import os
ROOT = '/content/drive/MyDrive/mcp-agent-attack-detection'
REVDIR = os.path.join(ROOT, 'experiments_R1', 'results')
os.environ.update(
    PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION='python', USE_TF='0', USE_FLAX='0',
    TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='1',
    R1_RESULTS_DIR=REVDIR,
)
os.chdir(os.path.join(ROOT, 'experiments_R1'))
import torch; print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
!python run_all.py

In [ ]:
# Cell 3 - consolidate the JSONs into experiments_R1/results/R1_RESULTS.md
import os, json
ROOT = '/content/drive/MyDrive/mcp-agent-attack-detection'
REVDIR = os.path.join(ROOT, 'experiments_R1', 'results')
def load(n):
    p = os.path.join(REVDIR, n)
    return json.load(open(p)) if os.path.exists(p) else None
ea, eb, ec, ed = load('e_a_confound.json'), load('e_b_edges.json'), load('e_c_sequence.json'), load('e_d_auprc.json')
L = ['# Paper 1 R1 experiments - Colab (final numbers)', '']
if ea:
    a1 = ea['a1_same_model']
    L += ['## E-A model-style confound',
          f"- A1 same-model (glm-only benign {a1['n_benign_glm']} + glm attacks): AUROC {a1['auroc'][0]:.4f} +/- {a1['auroc'][1]:.3f}, recall {a1['recall'][0]:.3f}, fpr {a1['fpr'][0]:.3f}",
          '- A2 per-model FPR probe: ' + ', '.join(f"{m}={d['fpr']:.3f}(n={d['n']})" for m, d in ea['a2_per_model_fpr'].items()),
          f"- A3 glm-vs-rest benign (task-disjoint): SBERT logreg {ea['a3_glm_vs_rest']['sbert']['logreg_auroc'][0]:.3f} / rf {ea['a3_glm_vs_rest']['sbert']['rf_auroc'][0]:.3f}; length-only rf {ea['a3_glm_vs_rest']['length_only']['rf_auroc'][0]:.3f}", '']
if eb:
    L += ['## E-B edge ablation', '| edges | AUROC | dataflow_edges | sessions w/ df | edges/node |', '|---|---|---|---|---|']
    for m, d in eb['edge_type'].items():
        au = d['metrics']['auroc']; frac = d.get('frac_sessions_with_dataflow', 0)
        L.append(f"| {m} | {au[0]:.4f} +/- {au[1]:.3f} | {d['n_dataflow_edges']} | {frac:.1%} | {d['total_edges']/max(d['total_nodes'],1):.2f} |")
    L += ['', '| token-len | AUROC | dataflow_edges |', '|---|---|---|']
    for m, d in eb['threshold'].items():
        au = d['metrics']['auroc']; L.append(f"| {m} | {au[0]:.4f} +/- {au[1]:.3f} | {d['n_dataflow_edges']} |")
    L.append('')
if ec:
    g = ec['gru']; L += ['## E-C GRU sequence baseline',
        f"- AUROC {g['auroc'][0]:.4f} +/- {g['auroc'][1]:.3f}, AUPRC {g['auprc'][0]:.4f}, benign-AP {g['auprc_benign'][0]:.4f}, recall {g['recall'][0]:.3f}, fpr {g['fpr'][0]:.3f}", '']
if ed:
    L += ['## E-D AUPRC (attack prevalence {:.3f})'.format(ed['prevalence_attack']),
          '| model | AUROC | AUPRC(attack) | AP(benign) | recall | fpr |', '|---|---|---|---|---|---|']
    for grp in ('architectures', 'classical'):
        for name, m in ed[grp].items():
            L.append(f"| {name} | {m['auroc'][0]:.4f} | {m['auprc'][0]:.4f} | {m['auprc_benign'][0]:.4f} | {m['recall'][0]:.3f} | {m['fpr'][0]:.3f} |")
    L.append('')
out = os.path.join(REVDIR, 'R1_RESULTS.md')
open(out, 'w').write('\n'.join(L))
print('wrote', out); print('\n'.join(L))